# When the Church Votes Left: How Progressive Bishops Supported the Workers' Party in Brazil

**Authors:** Tuñón, Guadalupe

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and load standard libraries.

In [ ]:
install.packages(c("tidyverse", "fixest", "modelsummary", "broom", "lmtest", "sandwich", "stargazer", "bigrquery", "DBI", "arrow"))

library(tidyverse)
library(fixest)
library(bigrquery)
library(DBI)

## Data Loading

Load datasets from BigQuery. The data is hosted in the PoliRep public dataset.

In [ ]:
# PoliRep public dataset — no change needed
project_id <- "polirep-app"

In [ ]:
# Load IPEA_electoral_data.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_ipea_electoral_data`"
ipea_electoral_data <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("ipea_electoral_data: %d rows, %d columns\n", nrow(ipea_electoral_data), ncol(ipea_electoral_data)))
head(ipea_electoral_data)

In [ ]:
# Load NIS_Bishop-class.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_nis_bishop_class`"
nis_bishop_class <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("nis_bishop_class: %d rows, %d columns\n", nrow(nis_bishop_class), ncol(nis_bishop_class)))
head(nis_bishop_class)

In [ ]:
# Load bishops_all_countries.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_bishops_all_countries`"
bishops_all_countries <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("bishops_all_countries: %d rows, %d columns\n", nrow(bishops_all_countries), ncol(bishops_all_countries)))
head(bishops_all_countries)

In [ ]:
# Load diodata.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_diodata`"
diodata <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("diodata: %d rows, %d columns\n", nrow(diodata), ncol(diodata)))
head(diodata)

In [ ]:
# Load muni_parish_database.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_muni_parish_database`"
muni_parish_database <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("muni_parish_database: %d rows, %d columns\n", nrow(muni_parish_database), ncol(muni_parish_database)))
head(muni_parish_database)

In [ ]:
# Load munidata.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_munidata`"
munidata <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("munidata: %d rows, %d columns\n", nrow(munidata), ncol(munidata)))
head(munidata)

In [ ]:
# Load panel_filiaco_mun_1970_pt.csv
sql <- "SELECT * FROM `polirep-app.paper_data.church_votes_left_panel_filiaco_mun_1970_pt`"
panel_filiaco_mun_1970_pt <- bq_project_query(project_id, sql) |> bq_table_download()
cat(sprintf("panel_filiaco_mun_1970_pt: %d rows, %d columns\n", nrow(panel_filiaco_mun_1970_pt), ncol(panel_filiaco_mun_1970_pt)))
head(panel_filiaco_mun_1970_pt)

## Write Data to Disk

Write BQ DataFrames as parquet files so downstream source scripts work unchanged.

In [ ]:
# Write BQ DataFrames to data/converted/ so source scripts work
dir.create("data/converted", recursive = TRUE, showWarnings = FALSE)
arrow::write_parquet(ipea_electoral_data, "data/converted/ipea_electoral_data.parquet")
arrow::write_parquet(nis_bishop_class, "data/converted/nis_bishop_class.parquet")
arrow::write_parquet(bishops_all_countries, "data/converted/bishops_all_countries.parquet")
arrow::write_parquet(diodata, "data/converted/diodata.parquet")
arrow::write_parquet(muni_parish_database, "data/converted/muni_parish_database.parquet")
arrow::write_parquet(munidata, "data/converted/munidata.parquet")
arrow::write_parquet(panel_filiaco_mun_1970_pt, "data/converted/panel_filiaco_mun_1970_pt.parquet")

cat("Wrote", length(list.files("data/converted", pattern = "\\.parquet$")), "parquet files to data/converted/\n")

## Data Preparation


### Load Primary Datasets

Load municipality characteristics, diocese characteristics, and electoral
outcome data from converted Parquet files.

In [ ]:
source("r_clean/scripts/00_setup.R")
munidata  <- readr::read_csv(here::here("data", "munidata.csv"), show_col_types = FALSE)
diodata   <- readr::read_csv(here::here("data", "diodata.csv"),  show_col_types = FALSE)
electoral <- readr::read_csv(here::here("data", "IPEA_electoral_data.csv"), show_col_types = FALSE)
cat("munidata:", nrow(munidata), "rows\n")
cat("diodata:", nrow(diodata), "rows\n")
cat("electoral:", nrow(electoral), "rows\n")


### Build Municipality-Election Panel

Merge municipality data with electoral outcomes on cod.2010, creating the
main analysis panel for electoral analysis (Tables 1, C1-C8).

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("munidata_panel_election:", nrow(munidata_panel_election), "rows\n")
cat("Elections:", unique(munidata_panel_election$election), "\n")
head(munidata_panel_election[, c("cod.2010", "election", "partido", "cargo", "votos", "vote.sh")], 10)


### Construct Electoral Treatment Variables

Create Retirement_Years (instrument) and Exposure (treatment) using censored
timing logic. If bishop retirement/appointment occurred before the election,
treatment equals years since 1978; otherwise censored at election year.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("Mean Retirement_Years:", mean(munidata_panel_election$Retirement_Years, na.rm=TRUE), "\n")
cat("Mean Exposure:", mean(munidata_panel_election$Exposure, na.rm=TRUE), "\n")
head(munidata_panel_election[, c("election", "I_YEAR", "JPIIAPT_YEAR", "Retirement_Years", "Exposure")], 10)


### Apply Electoral Sample Restrictions

Filter to PT presidential elections (1989, 1994, 1998, 2002) and exclude
archdioceses (CE_TYPE != 'A'). This produces the main analysis sample for
Table 1.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
electoral_sample <- munidata_panel_election %>%
  filter(partido == "PT", cargo == "Presidente",
         election %in% c(1989, 1994, 1998, 2002),
         CE_TYPE != "A")
cat("Analysis sample:", nrow(electoral_sample), "rows\n")
cat("Dioceses:", length(unique(electoral_sample$CE_1978)), "\n")
cat("Mean PT vote share:", mean(electoral_sample$vote.sh, na.rm=TRUE), "%\n")
table(electoral_sample$election)


### Build Diocese-Level Panel

Aggregate municipality votes to diocese level for Table C1 robustness check.
Groups by diocese, election, office, and party; sums votes and computes
diocese-level vote shares.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("diodata_panel_election:", nrow(diodata_panel_election), "rows\n")
cat("Dioceses:", length(unique(diodata_panel_election$CE_code)), "\n")
head(diodata_panel_election)


### Load Parish Turnover Data

Load parish-level data with yearbook observations. Used for Table 2 mechanism
analysis showing progressive bishops increased priest turnover.

In [ ]:
source("r_clean/scripts/00_setup.R")
muni_priests_data <- readr::read_csv(
  here::here("data", "muni_parish_database.csv"),
  show_col_types = FALSE
)
cat("Parish data:", nrow(muni_priests_data), "rows\n")
cat("Yearbooks:", unique(muni_priests_data$AC_year), "\n")
head(muni_priests_data)


### Prepare Parish Sample

Apply balanced parish panel filter (parishes exist in 1965, 1970, 1977),
exclude archdioceses, and construct binary treatment variables (I_cons,
cons) for Table 2.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("parish_panel:", nrow(parish_panel), "rows\n")
cat("Mean I_cons:", mean(parish_panel$I_cons, na.rm=TRUE), "\n")
cat("Mean cons:", mean(parish_panel$cons, na.rm=TRUE), "\n")
head(parish_panel[, c("AC_year", "I_YEAR", "JPIIAPT_YEAR", "I_cons", "cons", "turnover_priest_01")], 10)


### Load PT Local Presence Panel

Load PT affiliation panel tracking new member registrations by municipality
and year. Used for Table 3 mechanism analysis.

In [ ]:
source("r_clean/scripts/00_setup.R")
panel_filiacao_data <- readr::read_csv(
  here::here("data", "panel_filiaco_mun_1970_pt.csv"),
  show_col_types = FALSE
)
cat("PT panel:", nrow(panel_filiacao_data), "rows\n")
cat("Years:", range(panel_filiacao_data$year), "\n")
head(panel_filiacao_data)


### Prepare PT Panel Sample

Construct PT office activation indicator (threshold: 5+ new members) and
binary treatment variables. Excludes archdioceses and municipalities without
diocese linkage.

In [ ]:
source("r_clean/scripts/00_setup.R")
source("r_clean/scripts/01_data.R")
cat("pt_panel:", nrow(pt_panel), "rows\n")
cat("Mean had_office:", mean(pt_panel$had_office, na.rm=TRUE), "\n")
cat("Mean I_cons:", mean(pt_panel$I_cons, na.rm=TRUE), "\n")
head(pt_panel[, c("cod.1970", "year", "new_members", "year_start", "had_office", "I_cons", "cons")], 10)


## Data Loading and Validation


### Data Loading and Validation

Load 7 datasets (municipality, diocese, electoral, parish, PT party, bishop classification). Validate completeness and merge keys.

## Descriptive Analysis


### Descriptive Analysis

Generate maps showing PT territorial expansion (1989-2002) and mandated exposure to progressive bishops. Summary statistics table.

## Tests of Identification Design


### Tests of Identification Design

First stage visualization: scatter plot of mandated vs observed retirement, coefficient over time, F-statistics. Balance tests and placebo regression (1976 MDB vote).

## Main Electoral Analysis (Table 1)


### Main Electoral Analysis (Table 1)

2SLS and reduced-form estimates of progressive bishop exposure on PT presidential vote share. Four elections (1989, 1994, 1998, 2002), state FE, diocese clustering.

## Electoral Robustness Checks


### Electoral Robustness Checks

Diocese-level aggregation (C1), archdioceses included (C2), congressional elections (C3), bishop controls (C4), truncated sample (C5), categorical treatment (C6), drop conservatives (C7), alternative timing (C8), all left parties (D1).

## Mechanism 1 - Priest Turnover (Table 2)


### Mechanism 1 - Priest Turnover (Table 2)

Test whether progressive bishops increased priest turnover at the parish level. Six specifications: baseline, single-parish munis, secular parishes only, RS state progressives.

## Mechanism 2 - PT Local Presence (Table 3)


### Mechanism 2 - PT Local Presence (Table 3)

Test whether progressive bishops increased PT local party presence. Outcomes: office activation year, new members. Panel data 1970-2002.